# AI-Based Food Waste Prediction & Reduction System
Machine Learning PBL project

Team: Om Gorde (2124UMLM1100), Aayush Surwase (2124UMLM1058), Pranav Dange (2124UMLM1081), Dewang Gorhe (2124UMLM1007)

Dataset: Kaggle, *Global Food Wastage Dataset (2018-2024)*. Save the CSV as `data/global_food_wastage_dataset.csv` before running.

## 1. Dataset understanding
Inspect shape, dtypes, missing values, duplicates, years, countries and categories.

In [ ]:
%matplotlib inline
import os

# Work from the project root so data/ and the saved files resolve correctly.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd

DATA_PATH = "data/global_food_wastage_dataset.csv"   # adjust to your file name
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())
df.info()
print("Missing values:\n", df.isna().sum().sort_values(ascending=False))
print("Exact duplicate rows:", df.duplicated().sum())
print(df.describe().T)
print("Years:", sorted(df["Year"].unique()))
print("Countries:", df["Country"].nunique(),
      "| Food categories:", df["Food Category"].nunique())
print(df["Food Category"].value_counts())

## 2. Cleaning
Rename columns, convert types, remove duplicates and invalid rows. Record the before/after counts in the report.

In [ ]:
import numpy as np
import pandas as pd

# Rename columns by keyword so small header differences do not break the code.
RENAME_RULES = [
    ("total waste", "total_waste_tons"),
    ("economic loss", "economic_loss_musd"),
    ("per capita", "avg_waste_per_capita_kg"),
    ("population", "population_million"),
    ("household", "household_waste_pct"),
    ("food", "food_category"),
    ("country", "country"),
    ("year", "year"),
]

def rename_column(name):
    low = name.strip().lower()
    for key, new in RENAME_RULES:
        if key in low:
            return new
    return name.strip()

df = pd.read_csv("data/global_food_wastage_dataset.csv")
df.columns = [rename_column(c) for c in df.columns]
print("Renamed columns:", df.columns.tolist())
rows_loaded = len(df)

# Trim text so "Dairy " and "Dairy" are not treated as different classes.
for col in ["country", "food_category"]:
    df[col] = df[col].astype(str).str.strip()

# Convert numeric columns; unreadable values become NaN and are counted.
num_cols = ["year", "total_waste_tons", "economic_loss_musd",
            "avg_waste_per_capita_kg", "population_million",
            "household_waste_pct"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Exact duplicates removed:", df.duplicated().sum())
df = df.drop_duplicates().copy()

# Keep rows with a usable target and plausible values. Do not drop large
# values just because they look high: inspect them first.
before = len(df)
df = df.dropna(subset=["total_waste_tons"])
df = df[df["total_waste_tons"] >= 0]
df = df[df["household_waste_pct"].between(0, 100)
        | df["household_waste_pct"].isna()]
df = df[(df["population_million"] > 0) | df["population_million"].isna()]
df = df.copy()
print("Rows loaded:", rows_loaded, "| after cleaning:", len(df),
      "| removed by validity rules:", before - len(df))

## 3. Leakage check
A correlation near 1 between total waste and economic loss means economic loss must stay out of X. Per-capita waste is checked the same way.

In [ ]:
# Run after renaming columns (see section 2). Do this BEFORE choosing X.
print(df[["total_waste_tons", "economic_loss_musd"]].corr().round(3))

# Per-capita waste x population should reproduce total waste if it is derived.
implied_tons = df["avg_waste_per_capita_kg"] * df["population_million"] * 1000
print("Corr. with per-capita x population:",
      df["total_waste_tons"].corr(implied_tons).round(3))
print((df["total_waste_tons"] / implied_tons).describe())

## 4. Exploratory data analysis
Write your findings from these plots in the report (use real numbers; avoid causal claims).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print(df["total_waste_tons"].describe())
print("Zero-waste share:", (df["total_waste_tons"] == 0).mean().round(3))

sns.histplot(data=df, x="total_waste_tons", bins=40)
plt.title("Distribution of recorded total food waste")
plt.xlabel("Total waste (tons)")
plt.tight_layout(); plt.show()

cat_summary = (df.groupby("food_category")["total_waste_tons"]
               .agg(rows="count", median_tons="median", mean_tons="mean")
               .sort_values("median_tons", ascending=False))
print(cat_summary)
cat_summary["median_tons"].plot(kind="bar", figsize=(9, 3.5))
plt.title("Median recorded waste by food category")
plt.ylabel("Median waste (tons)"); plt.tight_layout(); plt.show()

sns.boxplot(data=df, x="food_category", y="total_waste_tons")
plt.xticks(rotation=25, ha="right")
plt.title("Total waste by food category")
plt.tight_layout(); plt.show()

country_summary = (df.groupby("country")["total_waste_tons"]
                   .agg(rows="count", median_tons="median")
                   .sort_values("median_tons", ascending=False))
print(country_summary.head(10))

yearly = df.groupby("year")["total_waste_tons"].agg(rows="count",
                                                     mean_tons="mean")
print(yearly)
yearly["mean_tons"].plot(marker="o", figsize=(6, 3.5))
plt.title("Mean recorded waste per record by year")
plt.ylabel("Mean waste (tons)"); plt.tight_layout(); plt.show()

corr_cols = ["total_waste_tons", "economic_loss_musd",
             "avg_waste_per_capita_kg", "population_million",
             "household_waste_pct", "year"]
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="Blues")
plt.title("Correlation matrix")
plt.tight_layout(); plt.show()

## 5. Feature engineering and model training
Inputs: year, population_million, household_waste_pct, country, food_category. Target: total_waste_tons. 80:20 split, 5-fold CV on the training set, selection by CV MAE.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

numeric = ["year", "population_million", "household_waste_pct"]
categorical = ["country", "food_category"]
X = df[numeric + categorical]
y = df["total_waste_tons"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]),
     categorical),
])
models = {
    "Dummy median": DummyRegressor(strategy="median"),
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=12, min_samples_leaf=5,
                                           random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=250,
                                           min_samples_leaf=2,
                                           random_state=42, n_jobs=-1),
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
results, best_model, best_name, best_cv = [], None, None, float("inf")
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    cv_mae = -cross_val_score(pipe, X_train, y_train, cv=cv,
                              scoring="neg_mean_absolute_error").mean()
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({"Model": name, "CV_MAE_tons": cv_mae,
                    "Test_MAE_tons": mean_absolute_error(y_test, pred),
                    "Test_RMSE_tons": mean_squared_error(y_test, pred) ** 0.5,
                    "Test_R2": r2_score(y_test, pred)})
    if cv_mae < best_cv:
        best_cv, best_model, best_name = cv_mae, pipe, name
print(pd.DataFrame(results).sort_values("CV_MAE_tons"))
print("Selected by CV MAE:", best_name)

### Optional robustness check (train on earlier years, test on the last year)

In [ ]:
# Optional robustness check: train on earlier years, test on the last year.
from sklearn.base import clone

last_year = df["year"].max()
early = df["year"] < last_year
check = clone(best_model).fit(X[early], y[early])
pred_last = check.predict(X[~early])
print("Time-split MAE (tons):", mean_absolute_error(y[~early], pred_last))
print("Time-split R2:", r2_score(y[~early], pred_last))

## 6. Evaluation
Largest errors, actual vs predicted, and feature importance (tree models).

In [ ]:
# Run after best_model has been selected
pred = best_model.predict(X_test)
errors = X_test.copy()
errors["actual"] = y_test
errors["predicted"] = pred
errors["absolute_error"] = (errors["actual"] - errors["predicted"]).abs()
print(errors.sort_values("absolute_error", ascending=False).head(10))
print("MAE as % of mean target:",
      round(100 * errors["absolute_error"].mean() / y_test.mean(), 1))

plt.scatter(y_test, pred, alpha=0.6)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(lims, lims, color="red", linewidth=1)
plt.xlabel("Actual waste (tons)"); plt.ylabel("Predicted waste (tons)")
plt.title("Actual vs predicted on the test set")
plt.tight_layout(); plt.show()

# Feature importance (only when the selected model is tree-based)
model_step = best_model.named_steps["model"]
if hasattr(model_step, "feature_importances_"):
    names = best_model.named_steps["preprocess"].get_feature_names_out()
    importance = pd.Series(model_step.feature_importances_, index=names)
    print(importance.sort_values(ascending=False).head(10))

## 7. Save model and reference files for the Streamlit app

In [ ]:
import json
import joblib

joblib.dump(best_model, "best_model.joblib")

# Defaults used by the app: typical population and household share per country.
country_ref = (df.groupby("country")[["population_million",
                                       "household_waste_pct"]]
               .median().reset_index())
country_ref["population_million"] = country_ref["population_million"].fillna(
    df["population_million"].median())
country_ref["household_waste_pct"] = country_ref["household_waste_pct"].fillna(
    df["household_waste_pct"].median())
country_ref.to_csv("country_reference.csv", index=False)

# Dataset-derived constant for the reduction scenario (USD lost per ton).
valid = df[(df["total_waste_tons"] > 0) & df["economic_loss_musd"].notna()]
loss_per_ton = float((valid["economic_loss_musd"] * 1e6
                      / valid["total_waste_tons"]).median())
with open("reduction_reference.json", "w") as f:
    json.dump({"median_loss_usd_per_ton": loss_per_ton}, f)
print("Saved model + reference files. Median loss per ton (USD):", loss_per_ton)